In [ ]:
!pip3 install pyspellchecker
import numpy as np
import pandas as pd
import re
import nltk
import nltk.data
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from spellchecker import SpellChecker
import os

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report



# removing stop words
with open('/content/drive/MyDrive/english', 'r') as file:
    stopwords = set(file.read().splitlines())
def process_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords]
    return ' '.join(filtered_words)


# word stemming
def stemming_words(text):
    stemmer = PorterStemmer()
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)


# spelling correction
spell_checker = SpellChecker()
def spelling_correction(text):
    words = text.split()
    corrected_words = [spell_checker.correction(word) if spell_checker.correction(word) else word for word in words]
    return ' '.join(corrected_words)

# removing all numbers
def remove_numbers(text):
    text = re.sub(r'[0-9]+', '', text)
    return text


# The list of chosen emoticons to be covered is based on manual review. It should include almost all frequent used emoticons in target dataset
emoticon_map = {
    # positive emojis
    ": )": " happy_emoji ",
    ": - )": " happy_emoji ",
    ": ]": " happy_emoji ",
    ": - ]": " happy_emoji ",
    ":d": " big_smile_emoji ",
    ": - D": " big_smile_emoji ",
    ":p": " tongue_out_emoji ",
    ": - P": " tongue_out_emoji ",
    "; )": " wink_emoji ",
    "; - )": " wink_emoji ",
    "; ]": " wink_emoji ",
    "; - ]": " wink_emoji ",
    "<3": " heart_emoji ",
    ":3": " happy_emoji ",
    ": - 3": " happy_emoji ",
    "xD": " laughing_emoji ",
    "XD": " laughing_emoji ",
    "= )": " happy_emoji ",
    "( :": " happy_emoji ",
    "( - :": " happy_emoji ",
    ": ' )": " happy_tear_emoji ",
    ": ' - )": " happy_tear_emoji ",
    "^ ^": " happy_emoji ",
    "^ - ^": " happy_emoji ",
    ": *": " kiss_emoji ",
    "xxx": " kiss_emoji ",
    "xx": " kiss_emoji ",
    ": - *": " kiss_emoji ",
    "> : D": " mischievous_emoji ",
    "> : 3": " mischievous_emoji ",
    "> : ]": " mischievous_emoji ",
    ": ^ )": " happy_emoji ",
    "^ _ ^": " happy_emoji ",
    "^ . ^": " happy_emoji ",
    "^ ^'": " embarrassed_emoji ",
    "^ ^ ;": " embarrassed_emoji ",
    "> : D <": " hug_emoji ",
    ": o )": " happy_emoji ",
    "* \ \ o / *": " cheer_emoji ",
    "o /": " cheer_emoji ",
    "\ \ o /": " cheer_emoji ",
    "> : D <": " hug_emoji ",
    ": - )": " happy_emoji ",
    ": }": " happy_emoji ",
    ": - }": " happy_emoji ",
    ": - ]": " happy_emoji ",
    ": - )": " happy_emoji ",
    ") ) )": " happy_emoji ",
    ") )": " happy_emoji ",
    ")": " happy_emoji ",
    ":/": " confused_emoji ",

    # negative emojis
    "( ( (": " sad_emoji ",
    "( (": " sad_emoji ",
    "(": " sad_emoji ",
    ": (": " sad_emoji ",
    ": - (": " sad_emoji ",
    ": [": " sad_emoji ",
    ": - [": " sad_emoji ",
    ": ' (": " cry_emoji ",
    ": ' - (": " cry_emoji ",
    "d:": " shocked_emoji ",
    "d8": " shocked_emoji ",
    "d;": " shocked_emoji ",
    "d=": " shocked_emoji ",
    "dx": " shocked_emoji ",
    "d-':": " shocked_emoji ",
    "> : (": " angry_emoji ",
    "> : - (": " angry_emoji ",
    "> : O": " surprised_emoji ",
    ": o": " surprised_emoji ",
    "> : 0": " surprised_emoji ",
    "> : |": " annoyed_emoji ",
    ": - /": " skeptical_emoji ",
    ": /": " skeptical_emoji ",
    ": - \\": " skeptical_emoji ",
    ": \\": " skeptical_emoji ",
    ": |": " neutral_emoji ",
    ": - |": " neutral_emoji ",
    "- _ -": " neutral_emoji ",
    "= _ =": " neutral_emoji ",
    "¬ _ ¬": " annoyed_emoji ",
    "- . -": " neutral_emoji ",
    ". _ .": " neutral_emoji ",
    "T _ T": " crying_emoji ",
    "T . T": " crying_emoji ",
    ": ' (": " cry_emoji ",
    "Q . Q": " crying_emoji ",
    "D : <": " angry_emoji ",
    "D - : <": " angry_emoji ",
    "D - :": " shocked_emoji ",
    "D : <": " angry_emoji ",
    "D 8 <": " angry_emoji ",
    "> _ <": " frustrated_emoji ",
    "> . <": " frustrated_emoji ",
    "> : )": " mischievous_emoji ",
    "> : D": " mischievous_emoji ",
}

emoticon_pattern = re.compile("|".join(re.escape(k) for k in emoticon_map.keys()))

def translate_emoticons(text):
    def replace(match):
        emoticon = match.group(0)
        return emoticon_map.get(emoticon, emoticon)
    return emoticon_pattern.sub(replace, text)

# Expanding contractions, standardizing slangs and translating emoticons to textual format
def process_contractions_slang_emoticon(text):
    re_map = [
        (r" ([a-z]+)'ll ", r" \1 will "),
        (r" ([a-z]+)'ve ", r" \1 have "),
        (r" ([a-z]+)'re ", r" \1 are "),
        (r" ([a-z]+)'d ", r" \1 would "),
        (r"\bcan't\b", "can not"),
        (r"\bcannot\b", "can not"),
        (r"\bwon't\b", "will not"),
        (r"\bdon't\b", "do not"),
        (r"\bdidn't\b", "did not"),
        (r"\bdoesn't\b", "does not"),
        (r"\bisn't\b", "is not"),
        (r"\baren't\b", "are not"),
        (r"\bwasn't\b", "was not"),
        (r"\bweren't\b", "were not"),
        (r"\bi'm\b", "i am"),
        (r"\bhe's\b", "he is"),
        (r"\bshe's\b", "she is"),
        (r"\bit's\b", "it is"),
        (r"\bthat's\b", "that is"),
        (r"\bthere's\b", "there is"),
        (r"\bwhere's\b", "where is"),
        (r"\bhere's\b", "here is"),
        (r"\bwho's\b", "who is"),
        (r"\bwhat's\b", "what is"),
        (r"\bhow's\b", "how is"),
        (r"\bwhen's\b", "when is"),
        (r"\bwhy's\b", "why is"),
        (r"\bdunno\b", "do not know"),
        (r"\bgonna\b", "going to"),
        (r"\bwanna\b", "want to"),
        (r"\bgotta\b", "got to"),
        (r"\blemme\b", "let me"),
        (r"\bkinda\b", "kind of"),
        (r"\bsorta\b", "sort of"),
        (r"\bhafta\b", "have to"),
        (r"\blotta\b", "lot of"),
        (r"\bdoin'\b", "doing"),
        (r"\btryin'\b", "doing"),
        (r"\bgotcha\b", "got you"),
        (r"\b'cause\b", "because"),
        (r"\bb'day\b", "birthday"),
        (r"\btbh\b", "to be honest"),
        (r"\bomw\b", "on my way"),
        (r"\bbrb\b", "be right back"),
        (r"\bidk\b", "i do not know"),
        (r"\bimo\b", "in my opinion"),
        (r"\bimho\b", "in my honest opinion"),
        (r"\bur\b", "your"),
        (r"\bu\b", "you"),
        (r"\bwassup\b", "what is up"),
        (r"\blotsa\b", "lots of"),
        (r"\bc'mon\b", "come on"),
        (r"\boughta\b", "ought to"),
        (r"\by'know\b", "you know"),
        (r"\bgimme\b", "give me"),
        (r"\by'all\b", "you all"),
        (r"\boutta\b", "out of"),
        (r"\bcoulda\b", "could have"),
        (r"\bwoulda\b", "would have"),
        (r"\bshoulda\b", "should have"),
        (r"\bmighta\b", "might have"),
        (r"\bthx\b", "thanks"),
        (r"\bsorri\b", "sorry"),
        (r"\bx\b", "kiss_emoji")

    ]

    for pattern, repl in re_map:
        text = re.sub(pattern, repl, text)

    text = translate_emoticons(text)

    return text


# lemmatization
lemmatizer = WordNetLemmatizer()
nltk.download('punkt')
nltk.download('wordnet')
def lemmatization(text):
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)


# Applying implemented preprocessing techniques
def clean_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'<url>', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = process_contractions_slang_emoticon(text)
    text = lemmatization(text)
    text = remove_numbers(text)
    text = re.sub(r'[.,<>&]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    # text = stemming_words(text)         not accurate, thus excluded
    # text = process_stopwords(text)      reduces accuracy, thus excluded
    # text = spelling_correction(text)    too slow & no significant improvement, thus excluded
    return text

# load raw data
with open('/content/drive/MyDrive/train_pos_full.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

with open('/content/drive/MyDrive/test_data.txt', 'r', encoding='utf-8') as file:
    test_content = file.readlines()



train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1

train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0

test = pd.DataFrame(test_content, columns=['tweet'])

train = pd.concat([train_pos, train_neg], ignore_index = True)


# preprocessing test dataset and store the resulting new dataset
test['tweet'] = test['tweet'].apply(clean_text)
test['tweet'].to_csv('/content/drive/MyDrive/test_cleaned.txt', index=False, header=False)


# preprocessing training datasets(both positive and negative) and store the resulting new datasets
train_pos['tweet'] = train_pos['tweet'].apply(clean_text)
train_neg['tweet'] = train_neg['tweet'].apply(clean_text)
train_pos['tweet'].to_csv('/content/drive/MyDrive/train_pos_full_cleaned.txt', index=False, header=False)
train_neg['tweet'].to_csv('/content/drive/MyDrive/train_neg_full_cleaned.txt', index=False, header=False)

# Note that in all models, we always load the preprocessed datasets from google drive. For this reason, please make sure that they are in the drive
#by running this code. Optionally, we also provide preprocessed datasets directly in our submission.

#3 Optionally, just preprocess the training datasets and do not store the preprocessed datasets
#train['tweet'] = train['tweet'].apply(clean_text)

print("Preprocessing done")